# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

# 1. Initial Setup

In [11]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 10.9062


In [13]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 251.77 GB
MemFree: 59.08 GB
MemAvailable: 235.00 GB
Free GPU Memory (GB): 10.9062

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA GeForce GTX 1080 Ti

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been sa

# 2. Debug HQQ for high accuracy

In [6]:
import os
import pandas as pd
import glob
import re
import random

def extract_config(filename):
    pattern = r"(.+?)_(.+?)_(.+?)_typo(\d+)_(.+?)_tok(\d+)_temp([\d.]+)_(.+?)_rep(\d+)_beams(\d+)_maxent(.+?)_rows(\d+)"
    match = re.match(pattern, filename)
    if match:
        return {
            'model_name': match.group(1),
            'dataset_name': match.group(2),
            'typo_type': match.group(3),
            'typo_intensity': match.group(4),
            'beam_search_str': match.group(5),
            'max_new_tokens': match.group(6),
            'temperature': match.group(7),
            'strategy_str': match.group(8),
            'n_repeats': match.group(9),
            'n_beams': match.group(10),
            'max_entries': match.group(11),
            'num_excel_rows': match.group(12)
        }
    return None

def compare_excel_files(file1, file2, config, max_rows=100):
    df1 = pd.read_excel(file1, nrows=max_rows)
    df2 = pd.read_excel(file2, nrows=max_rows)
    
    mismatches = []
    
    for i in range(min(len(df1), len(df2))):
        if df1.loc[i, 'Is Correct'] != df2.loc[i, 'Is Correct']:
            mismatch = {
                'Query ID': df1.loc[i, 'Query ID'],
                'Query': df1.loc[i, 'Query'],
                'Answer': df1.loc[i, 'Answer'],
                f"{config['model1']}_Is Correct": df1.loc[i, 'Is Correct'],
                f"{config['model1']}_Generated Response": df1.loc[i, 'Generated Response'],
                f"{config['model1']}_Token Probabilities": df1.loc[i, 'Token Probabilities'],
                f"{config['model2']}_Is Correct": df2.loc[i, 'Is Correct'],
                f"{config['model2']}_Generated Response": df2.loc[i, 'Generated Response'],
                f"{config['model2']}_Token Probabilities": df2.loc[i, 'Token Probabilities'],
            }
            mismatches.append({**config, **mismatch})
    
    return mismatches

def main(exp_id, model1, model2, num_configs=10):
    results_path = "/nfs/homedirs/daro/git/quantization-reliability/results"
    exp_dir = os.path.join(results_path, "reliability_eval", f"reliability_eval_{exp_id}")
    debug_dir = os.path.join(results_path, "reliability_eval", "debugging", exp_id)
    os.makedirs(debug_dir, exist_ok=True)
    
    all_files = glob.glob(os.path.join(exp_dir, f"*raw_table*.xlsx"))
    
    configs = []
    for file in all_files:
        config = extract_config(os.path.basename(file))
        if config and config['model_name'] == model1:  # Only consider configs for model1
            config.pop('model_name')  # Remove model_name from config
            configs.append(config)
    
    # Randomly select configurations
    selected_configs = random.sample(configs, min(num_configs, len(configs)))
    
    all_mismatches = []
    
    for config in selected_configs:
        file1 = next((f for f in all_files if extract_config(os.path.basename(f)) == {**config, 'model_name': model1}), None)
        file2 = next((f for f in all_files if extract_config(os.path.basename(f)) == {**config, 'model_name': model2}), None)
        
        if file1 and file2:
            config_with_models = {**config, 'model1': model1, 'model2': model2}
            mismatches = compare_excel_files(file1, file2, config_with_models)
            all_mismatches.extend(mismatches)
    
    if all_mismatches:
        output_file = os.path.join(debug_dir, f"mismatches_{model1}_vs_{model2}_{exp_id}.xlsx")
        pd.DataFrame(all_mismatches).to_excel(output_file, index=False)
        print(f"Mismatches saved to: {output_file}")
    else:
        print("No mismatches found.")

if __name__ == "__main__":
    exp_id = "pert-awq-bnb-hqq-10-07"
    model1 = "Llama-3-8B"
    model2 = "Llama-3-8B-HQQ-mixed-local"
    main(exp_id, model1, model2)

Mismatches saved to: /nfs/homedirs/daro/git/quantization-reliability/results/reliability_eval/debugging/pert-awq-bnb-hqq-10-07/mismatches_Llama-3-8B_vs_Llama-3-8B-HQQ-mixed-local_pert-awq-bnb-hqq-10-07.xlsx


### Calculate number of correct rows

In [8]:
import os
import pandas as pd
import glob
import re

def extract_config(filename):
    pattern = r"(.+?)_(.+?)_(.+?)_typo(\d+)_(.+?)_tok(\d+)_temp([\d.]+)_(.+?)_rep(\d+)_beams(\d+)_maxent(.+?)_rows(\d+)"
    match = re.match(pattern, filename)
    if match:
        return {
            'model_name': match.group(1),
            'dataset_name': match.group(2),
            'typo_type': match.group(3),
            'typo_intensity': match.group(4),
            'beam_search_str': match.group(5),
            'max_new_tokens': match.group(6),
            'temperature': match.group(7),
            'strategy_str': match.group(8),
            'n_repeats': match.group(9),
            'n_beams': match.group(10),
            'max_entries': match.group(11),
            'num_excel_rows': match.group(12)
        }
    return None

def count_correct_predictions(exp_id, model_name, typo_type='char_insertion'):
    results_path = "/nfs/homedirs/daro/git/quantization-reliability/results"
    exp_dir = os.path.join(results_path, "reliability_eval", f"reliability_eval_{exp_id}")
    
    files = glob.glob(os.path.join(exp_dir, f"{model_name}*raw_table*.xlsx"))
    
    results = {}
    
    for file in files:
        config = extract_config(os.path.basename(file))
        if config and config['typo_type'] == typo_type:
            df = pd.read_excel(file)
            dataset = config['dataset_name']
            if dataset not in results:
                results[dataset] = {'correct': 0, 'total': 0}
            results[dataset]['correct'] += df['Is Correct'].sum()
            results[dataset]['total'] += len(df)
    
    return results

def main(exp_id, model1, model2):
    print(f"Analyzing experiment: {exp_id}")
    print(f"Comparing models: {model1} vs {model2}")
    print(f"Filtering for typo_type: char_insertion")
    
    results1 = count_correct_predictions(exp_id, model1)
    results2 = count_correct_predictions(exp_id, model2)
    
    all_datasets = set(list(results1.keys()) + list(results2.keys()))
    
    for dataset in all_datasets:
        print(f"\nResults for dataset: {dataset}")
        
        correct1 = results1.get(dataset, {'correct': 0, 'total': 0})['correct']
        total1 = results1.get(dataset, {'correct': 0, 'total': 0})['total']
        correct2 = results2.get(dataset, {'correct': 0, 'total': 0})['correct']
        total2 = results2.get(dataset, {'correct': 0, 'total': 0})['total']
        
        accuracy1 = correct1 / total1 if total1 > 0 else 0
        accuracy2 = correct2 / total2 if total2 > 0 else 0
        
        print(f"{model1}:")
        print(f"  Total correct predictions: {correct1}")
        print(f"  Total rows: {total1}")
        print(f"  Accuracy: {accuracy1:.4f}")
        
        print(f"\n{model2}:")
        print(f"  Total correct predictions: {correct2}")
        print(f"  Total rows: {total2}")
        print(f"  Accuracy: {accuracy2:.4f}")
        
        print(f"\nAccuracy difference ({model2} - {model1}): {accuracy2 - accuracy1:.4f}")
    
    # Calculate and print overall results
    total_correct1 = sum(results['correct'] for results in results1.values())
    total_rows1 = sum(results['total'] for results in results1.values())
    total_correct2 = sum(results['correct'] for results in results2.values())
    total_rows2 = sum(results['total'] for results in results2.values())
    
    overall_accuracy1 = total_correct1 / total_rows1 if total_rows1 > 0 else 0
    overall_accuracy2 = total_correct2 / total_rows2 if total_rows2 > 0 else 0
    
    print("\nOverall Results:")
    print(f"{model1} overall accuracy: {overall_accuracy1:.4f}")
    print(f"{model2} overall accuracy: {overall_accuracy2:.4f}")
    print(f"Overall accuracy difference ({model2} - {model1}): {overall_accuracy2 - overall_accuracy1:.4f}")

if __name__ == "__main__":
    exp_id = "pert-awq-bnb-hqq-10-07"
    model1 = "Llama-3-8B"
    model2 = "Llama-3-8B-HQQ-mixed-local"
    main(exp_id, model1, model2)

Analyzing experiment: pert-awq-bnb-hqq-10-07
Comparing models: Llama-3-8B vs Llama-3-8B-HQQ-mixed-local
Filtering for typo_type: char_insertion

Results for dataset: P30
Llama-3-8B:
  Total correct predictions: 650
  Total rows: 1200
  Accuracy: 0.5417

Llama-3-8B-HQQ-mixed-local:
  Total correct predictions: 146
  Total rows: 300
  Accuracy: 0.4867

Accuracy difference (Llama-3-8B-HQQ-mixed-local - Llama-3-8B): -0.0550

Results for dataset: P740
Llama-3-8B:
  Total correct predictions: 455
  Total rows: 1200
  Accuracy: 0.3792

Llama-3-8B-HQQ-mixed-local:
  Total correct predictions: 88
  Total rows: 300
  Accuracy: 0.2933

Accuracy difference (Llama-3-8B-HQQ-mixed-local - Llama-3-8B): -0.0858

Results for dataset: P364
Llama-3-8B:
  Total correct predictions: 688
  Total rows: 1200
  Accuracy: 0.5733

Llama-3-8B-HQQ-mixed-local:
  Total correct predictions: 155
  Total rows: 300
  Accuracy: 0.5167

Accuracy difference (Llama-3-8B-HQQ-mixed-local - Llama-3-8B): -0.0567

Results for da

### calculate accuracies for each dataset

In [10]:
import os
import pandas as pd
import glob
import re

def extract_config(filename):
    pattern = r"(.+?)_(.+?)_(.+?)_typo(\d+)_(.+?)_tok(\d+)_temp([\d.]+)_(.+?)_rep(\d+)_beams(\d+)_maxent(.+?)_rows(\d+)"
    match = re.match(pattern, filename)
    if match:
        return {
            'model_name': match.group(1),
            'dataset_name': match.group(2),
            'typo_type': match.group(3),
            'typo_intensity': match.group(4),
            'beam_search_str': match.group(5),
            'max_new_tokens': match.group(6),
            'temperature': match.group(7),
            'strategy_str': match.group(8),
            'n_repeats': match.group(9),
            'n_beams': match.group(10),
            'max_entries': match.group(11),
            'num_excel_rows': match.group(12)
        }
    return None

def count_correct_predictions(exp_id, model_name, typo_type='char_insertion'):
    results_path = "/nfs/homedirs/daro/git/quantization-reliability/results"
    exp_dir = os.path.join(results_path, "reliability_eval", f"reliability_eval_{exp_id}")
    
    files = glob.glob(os.path.join(exp_dir, f"{model_name}*raw_table*.xlsx"))
    
    results = {}
    
    for file in files:
        config = extract_config(os.path.basename(file))
        if config and config['typo_type'] == typo_type:
            df = pd.read_excel(file)
            dataset = config['dataset_name']
            if dataset not in results:
                results[dataset] = {'correct': 0, 'total': 0}
            results[dataset]['correct'] += df['Is Correct'].sum()
            results[dataset]['total'] += len(df)
    
    return results

def main(exp_id, model1, model2):
    print(f"Analyzing experiment: {exp_id}")
    print(f"Comparing models: {model1} vs {model2}")
    print(f"Filtering for typo_type: char_insertion")
    
    results1 = count_correct_predictions(exp_id, model1)
    results2 = count_correct_predictions(exp_id, model2)
    
    all_datasets = set(list(results1.keys()) + list(results2.keys()))
    
    for dataset in all_datasets:
        print(f"\nResults for dataset: {dataset}")
        
        correct1 = results1.get(dataset, {'correct': 0, 'total': 0})['correct']
        total1 = results1.get(dataset, {'correct': 0, 'total': 0})['total']
        correct2 = results2.get(dataset, {'correct': 0, 'total': 0})['correct']
        total2 = results2.get(dataset, {'correct': 0, 'total': 0})['total']
        
        accuracy1 = correct1 / total1 if total1 > 0 else 0
        accuracy2 = correct2 / total2 if total2 > 0 else 0
        
        print(f"{model1}:")
        print(f"  Total correct predictions: {correct1}")
        print(f"  Total rows: {total1}")
        print(f"  Accuracy: {accuracy1:.4f}")
        
        print(f"\n{model2}:")
        print(f"  Total correct predictions: {correct2}")
        print(f"  Total rows: {total2}")
        print(f"  Accuracy: {accuracy2:.4f}")
        
        print(f"\nAccuracy difference ({model2} - {model1}): {accuracy2 - accuracy1:.4f}")
    
    # Calculate and print overall results
    total_correct1 = sum(results['correct'] for results in results1.values())
    total_rows1 = sum(results['total'] for results in results1.values())
    total_correct2 = sum(results['correct'] for results in results2.values())
    total_rows2 = sum(results['total'] for results in results2.values())
    
    overall_accuracy1 = total_correct1 / total_rows1 if total_rows1 > 0 else 0
    overall_accuracy2 = total_correct2 / total_rows2 if total_rows2 > 0 else 0
    
    print("\nOverall Results:")
    print(f"{model1} overall accuracy: {overall_accuracy1:.4f}")
    print(f"{model2} overall accuracy: {overall_accuracy2:.4f}")
    print(f"Overall accuracy difference ({model2} - {model1}): {overall_accuracy2 - overall_accuracy1:.4f}")

if __name__ == "__main__":
    exp_id = "pert-awq-bnb-hqq-10-07"
    model1 = "Llama-3-8B"
    model2 = "Llama-3-8B-HQQ-mixed-local"
    main(exp_id, model1, model2)

Analyzing experiment: pert-awq-bnb-hqq-10-07
Comparing models: Llama-3-8B vs Llama-3-8B-HQQ-mixed-local
Filtering for typo_type: char_insertion

Results for dataset: P30
Llama-3-8B:
  Total correct predictions: 650
  Total rows: 1200
  Accuracy: 0.5417

Llama-3-8B-HQQ-mixed-local:
  Total correct predictions: 146
  Total rows: 300
  Accuracy: 0.4867

Accuracy difference (Llama-3-8B-HQQ-mixed-local - Llama-3-8B): -0.0550

Results for dataset: P740
Llama-3-8B:
  Total correct predictions: 455
  Total rows: 1200
  Accuracy: 0.3792

Llama-3-8B-HQQ-mixed-local:
  Total correct predictions: 88
  Total rows: 300
  Accuracy: 0.2933

Accuracy difference (Llama-3-8B-HQQ-mixed-local - Llama-3-8B): -0.0858

Results for dataset: P364
Llama-3-8B:
  Total correct predictions: 688
  Total rows: 1200
  Accuracy: 0.5733

Llama-3-8B-HQQ-mixed-local:
  Total correct predictions: 155
  Total rows: 300
  Accuracy: 0.5167

Accuracy difference (Llama-3-8B-HQQ-mixed-local - Llama-3-8B): -0.0567

Results for da

### Calculate excel number of rows

In [14]:
import os
import pandas as pd
import glob
import re
from collections import defaultdict

def extract_config(filename):
    pattern = r"(.+?)_(.+?)_(.+?)_typo(\d+)_(.+?)_tok(\d+)_temp([\d.]+)_(.+?)_rep(\d+)_beams(\d+)_maxent(.+?)_rows(\d+)"
    match = re.match(pattern, filename)
    if match:
        return {
            'model_name': match.group(1),
            'dataset_name': match.group(2),
            'typo_type': match.group(3),
            'typo_intensity': match.group(4),
            'beam_search_str': match.group(5),
            'max_new_tokens': match.group(6),
            'temperature': match.group(7),
            'strategy_str': match.group(8),
            'n_repeats': match.group(9),
            'n_beams': match.group(10),
            'max_entries': match.group(11),
            'num_excel_rows': match.group(12)
        }
    return None

def analyze_excel_files(exp_id):
    results_path = "/nfs/homedirs/daro/git/quantization-reliability/results"
    exp_dir = os.path.join(results_path, "reliability_eval", f"reliability_eval_{exp_id}")
    
    files = glob.glob(os.path.join(exp_dir, "*raw_table*.xlsx"))
    
    row_counts = defaultdict(list)
    
    for file in files:
        config = extract_config(os.path.basename(file))
        if config:
            df = pd.read_excel(file)
            row_count = len(df)
            config['actual_rows'] = row_count
            row_counts[row_count].append(config)
    
    return row_counts

def print_analysis(row_counts):
    for row_count, configs in sorted(row_counts.items()):
        print(f"\nNumber of rows: {row_count}")
        print(f"Number of combinations: {len(configs)}")
        print("Configurations:")
        for config in configs:
            print(f"  - Model: {config['model_name']}")
            print(f"    Dataset: {config['dataset_name']}")
            print(f"    Typo Type: {config['typo_type']}")
            print(f"    Typo Intensity: {config['typo_intensity']}")
            print(f"    Beam Search: {config['beam_search_str']}")
            print(f"    Max New Tokens: {config['max_new_tokens']}")
            print(f"    Temperature: {config['temperature']}")
            print(f"    Strategy: {config['strategy_str']}")
            print(f"    Repeats: {config['n_repeats']}")
            print(f"    Beams: {config['n_beams']}")
            print(f"    Max Entries: {config['max_entries']}")
            print(f"    Specified Excel Rows: {config['num_excel_rows']}")
            print("    ---")

def main(exp_id):
    print(f"Analyzing experiment: {exp_id}")
    
    row_counts = analyze_excel_files(exp_id)
    print_analysis(row_counts)

if __name__ == "__main__":
    exp_id = "pert-awq-bnb-hqq-10-07"
    main(exp_id)

Analyzing experiment: pert-awq-bnb-hqq-10-07

Number of rows: 100
Number of combinations: 4800
Configurations:
  - Model: Llama-3-8B-AWQ-4bit-local
    Dataset: P27
    Typo Type: word_taxonomy_neg
    Typo Intensity: 1
    Beam Search: sample
    Max New Tokens: 25
    Temperature: 0.1
    Strategy: direct_completion
    Repeats: 1
    Beams: 5
    Max Entries: all
    Specified Excel Rows: 100
    ---
  - Model: Llama-3-8B-AWQ-4bit-local
    Dataset: P264
    Typo Type: char_substitution
    Typo Intensity: 2
    Beam Search: sample
    Max New Tokens: 25
    Temperature: 0.1
    Strategy: direct_completion
    Repeats: 1
    Beams: 5
    Max Entries: all
    Specified Excel Rows: 100
    ---
  - Model: Llama-3-8B
    Dataset: P159
    Typo Type: word_taxonomy_pos
    Typo Intensity: 1
    Beam Search: sample
    Max New Tokens: 25
    Temperature: 0.1
    Strategy: direct_completion
    Repeats: 1
    Beams: 5
    Max Entries: all
    Specified Excel Rows: 100
    ---
  - Model: Lla